# Numerisk integrasjon

```{admonition} Læringsutbytte
Etter å ha arbeidet med dette temaet, skal du kunne:

1. forklare hvordan et bestemt integral kan tilnærmes med summer
2. implementere venstre-, høyre- og midtpunktstilnærming
3. forklare og implementere trapesmetoden
4. undersøke hvordan antall delintervaller påvirker feilen
5. integrere funksjoner og diskrete måledata numerisk
6. bruke moderne integrasjonsfunksjoner fra NumPy og SciPy
7. anvende numerisk integrasjon på kjemiske data, som kromatogrammer og spektre
```

## Integrasjon i kjemi

Et bestemt integral kan tolkes som et samlet bidrag over et intervall. I kjemi møter vi slike integraler blant annet når vi:

- finner arealet under en kromatografisk topp
- integrerer signaler i NMR-spektre
- beregner total stoffmengde fra en tidsavhengig strøm eller reaksjonsfart
- løser differensiallikninger

Numerisk integrasjon er spesielt nyttig når vi **ikke har en analytisk funksjon**, men bare diskrete måledata.

## Fra areal til sum

For en funksjon $f(x)$ på intervallet $[a,b]$ deler vi intervallet i $n$ biter med bredde

$$h=\frac{b-a}{n}.$$

Vi kan så summere arealet av enkle geometriske figurer.


## Rektangelmetoden

Med venstre endepunkt som høyde får vi

$$\int_a^b f(x)\,dx\approx h\sum_{k=0}^{n-1}f(a+kh).$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rectangle_left(f, a, b, n):
    h = (b - a) / n
    area = 0.0

    for k in range(n):
        x = a + k*h
        area += f(x) * h

    return area

def f(x):
    return x**3

numerical = rectangle_left(f, 0, 5, 1000)
analytical = 5**4 / 4 - 0**4 / 4

print("Numerical:", numerical)
print("Analytical:", analytical)


Her bruker både den numeriske og analytiske kontrollen **samme integrasjonsgrenser**.

Høyretilnærmingen bruker høyre endepunkt, mens midtpunktstilnærmingen bruker sentrum i hvert delintervall:

$$x_{\mathrm{midt},k}=a+\left(k+\frac12\right)h.$$


In [ ]:
def rectangle_right(f, a, b, n):
    h = (b - a) / n
    area = 0.0

    for k in range(n):
        x = a + (k + 1)*h
        area += f(x) * h

    return area

def rectangle_midpoint(f, a, b, n):
    h = (b - a) / n
    area = 0.0

    for k in range(n):
        x_mid = a + (k + 0.5)*h
        area += f(x_mid) * h

    return area


## Trapesmetoden

I stedet for å anta at funksjonen er konstant i hvert delintervall, kan vi forbinde nabopunktene med en rett linje. Arealet blir da et trapes:

$$A_k=\frac{f(x_k)+f(x_{k+1})}{2}h.$$

Summert over hele intervallet:

$$\int_a^b f(x)\,dx\approx
h\left[\frac{f(x_0)}2+\sum_{k=1}^{n-1}f(x_k)+\frac{f(x_n)}2\right].$$


In [ ]:
def trapezoid_method(f, a, b, n):
    h = (b - a) / n
    area = 0.5 * (f(a) + f(b))

    for k in range(1, n):
        area += f(a + k*h)

    return area * h

print(trapezoid_method(f, 0, 5, 100))


## Feil og konvergens

En numerisk verdi bør ledsages av en kontroll. En enkel strategi er å øke antall delintervaller og se om resultatet stabiliserer seg.


In [ ]:
exact = 5**4 / 4

for n in [5, 10, 50, 100, 1000]:
    result = trapezoid_method(f, 0, 5, n)
    error = abs(result - exact)
    print(f"n={n:4d}  integral={result:10.6f}  error={error:.3e}")


## Ferdige funksjoner

Når vi har forstått algoritmene, bruker vi vanligvis bibliotekfunksjoner.

For diskrete datapunkter kan vi bruke `scipy.integrate.trapezoid` og `scipy.integrate.simpson`. For en funksjon kan vi bruke adaptiv kvadratur med `scipy.integrate.quad`.


In [ ]:
from scipy import integrate

x = np.linspace(0, 5, 101)
y = f(x)

trap = integrate.trapezoid(y, x=x)
simp = integrate.simpson(y, x=x)
quad_value, quad_error = integrate.quad(f, 0, 5)

print("Trapezoid:", trap)
print("Simpson:", simp)
print("quad:", quad_value)
print("Estimated quad error:", quad_error)


## Kjemisk eksempel: arealet av en kromatografisk topp

Et kromatogram består av målte signalverdier ved bestemte tider. Da finnes det ikke nødvendigvis en funksjon vi kan antiderivere. Trapesmetoden passer derfor svært godt.

Vi lager et lite syntetisk datasett som representerer én kromatografisk topp:


In [ ]:
time = np.array([4.0, 4.2, 4.4, 4.6, 4.8, 5.0, 5.2, 5.4, 5.6, 5.8, 6.0])
signal = np.array([0.01, 0.04, 0.16, 0.50, 0.88, 1.00, 0.82, 0.46, 0.17, 0.05, 0.01])

peak_area = integrate.trapezoid(signal, x=time)

plt.plot(time, signal, "o-")
plt.fill_between(time, signal, alpha=0.25)
plt.xlabel("Retention time (min)")
plt.ylabel("Detector signal")
plt.show()

print(f"Peak area = {peak_area:.3f} signal·min")


Arealet kan for eksempel være proporsjonalt med stoffmengden eller konsentrasjonen etter en kalibrering. Dette er et godt eksempel på hvorfor numerisk integrasjon av **data** er minst like viktig som numerisk integrasjon av funksjoner.

Du kan utforske trapesintegrasjon av kromatogrammet i Basthon:

<iframe src="../../basthon/?from=examples/numerical_chromatogram_integration.py" width="100%" height="650" frameborder="0" title="Basthon: numerisk integrasjon av kromatogram" loading="lazy" allowfullscreen></iframe>

## Oppgaver

```{admonition} Oppgave 1 – rektangelmetodene
:class: tip
Integrer $f(x)=x^2-2x+4$ fra 2 til 8 med venstre-, høyre- og midtpunktstilnærming. Bruk først $n=10$ og deretter $n=100$. Sammenlikn med den analytiske verdien.
```

```{admonition} Oppgave 2 – konvergens
:class: tip
Lag et plott av absolutt feil som funksjon av $n$ for trapesmetoden. Hva skjer når $n$ øker?
```

```{admonition} Oppgave 3 – bibliotek
:class: tip
Integrer $f(x)=e^{-x^2}$ fra 0 til 2 med `trapezoid`, `simpson` og `quad`. Sammenlikn svarene.
```

```{admonition} Oppgave 4 – kromatogram
:class: tip
Beregn arealet under en kromatografisk topp fra et sett med diskrete tids- og signalverdier. Sammenlikn din egen trapesmetode med `scipy.integrate.trapezoid`.
```

```{admonition} Oppgave 5 – baseline
:class: tip
Et kromatogram har en konstant baseline på 0,08 signalenheter. Beregn topparealet både med og uten å trekke fra baseline. Hvor stor betydning får dette?
```

```{admonition} Oppgave 6 – reaksjonsfart
:class: tip
En reaksjonsfart $v(t)$ er målt ved flere tidspunkter. Forklar hvorfor integralet $\\int v(t)dt$ kan knyttes til hvor mye stoff som er omsatt. Lag et lite eksempel med egne data.
```

```{admonition} Oppgave 7 – vanskelige funksjoner
:class: tip
Finn en funksjon med sterk krumning eller en smal topp. Undersøk hvor mange delintervaller rektangel- og trapesmetoden trenger for å gi et stabilt resultat.
```
